# 04 — Ensemble & advanced models: XGBoost, LightGBM, Prophet

**Goal:** stronger learners. XGBoost & LightGBM use the full feature set to predict the next-day log-return (tuned with time-series CV); Prophet is an additive trend/seasonality forecaster on the price. All evaluated on the same test set/metrics.

In [1]:
import sys, os, warnings
warnings.filterwarnings("ignore")
# make the models/ folder importable whether the notebook runs from models/ or the repo root
HERE = os.getcwd()
MODELS_DIR = HERE if os.path.exists(os.path.join(HERE, "utils.py")) else os.path.join(HERE, "models")
sys.path.insert(0, MODELS_DIR)
import utils
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

## 1. Data and split

In [2]:
df = utils.load_modeling_frame()
train, val, test = utils.chrono_split(df)
trainval = pd.concat([train, val])
Xtr, ytr = trainval[utils.FEATURE_COLUMNS], trainval[utils.TARGET]
Xte = test[utils.FEATURE_COLUMNS]
actual = utils.actual_test_price(test)
tscv = TimeSeriesSplit(n_splits=5)

## 2. XGBoost (randomized search, time-series CV)

In [3]:
xgb_grid = {"n_estimators": [200, 400], "max_depth": [2, 3, 4],
            "learning_rate": [0.03, 0.1], "subsample": [0.8, 1.0]}
xgb_cv = RandomizedSearchCV(xgb.XGBRegressor(random_state=utils.SEED, n_jobs=-1),
                            xgb_grid, n_iter=8, cv=tscv,
                            scoring="neg_root_mean_squared_error", random_state=utils.SEED)
xgb_cv.fit(Xtr, ytr)
print("XGB best:", xgb_cv.best_params_)
xgb_pred = utils.reconstruct_price(test["y"], xgb_cv.predict(Xte))
xgb_m = utils.evaluate(actual, xgb_pred)
print("XGBoost:", {k: round(v, 4) for k, v in xgb_m.items()})
utils.save_predictions("xgboost", test, xgb_pred);

XGB best: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 2, 'learning_rate': 0.03}
XGBoost: {'MAE': 1.7727, 'RMSE': 2.4528, 'MAPE': 1.3691, 'R2': 0.9896}


## 3. LightGBM (randomized search, time-series CV)

In [4]:
lgb_grid = {"n_estimators": [200, 400], "num_leaves": [15, 31],
            "learning_rate": [0.03, 0.1], "subsample": [0.8, 1.0]}
lgb_cv = RandomizedSearchCV(
    lgb.LGBMRegressor(random_state=utils.SEED, n_jobs=-1, verbose=-1, subsample_freq=1),
    lgb_grid, n_iter=8, cv=tscv, scoring="neg_root_mean_squared_error",
    random_state=utils.SEED)
lgb_cv.fit(Xtr, ytr)
print("LGBM best:", lgb_cv.best_params_)
lgb_pred = utils.reconstruct_price(test["y"], lgb_cv.predict(Xte))
lgb_m = utils.evaluate(actual, lgb_pred)
print("LightGBM:", {k: round(v, 4) for k, v in lgb_m.items()})
utils.save_predictions("lightgbm", test, lgb_pred);

LGBM best: {'subsample': 0.8, 'num_leaves': 15, 'n_estimators': 200, 'learning_rate': 0.03}
LightGBM: {'MAE': 1.4872, 'RMSE': 2.1848, 'MAPE': 1.1478, 'R2': 0.9917}


## 4. Prophet (one-step walk-forward)
Prophet models the price as trend + seasonality. We run it **one-step-ahead** (refit on a rolling window each day) so it is comparable to the others. If Prophet is not installed, this section is skipped.

In [5]:
try:
    from prophet import Prophet
    import logging
    logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
    logging.getLogger("prophet").setLevel(logging.ERROR)
    have_prophet = True
except Exception as e:
    print("Prophet unavailable -> skipping:", e)
    have_prophet = False

Importing plotly failed. Interactive plots will not work.


In [6]:
prophet_m = None
if have_prophet:
    dfp = df[["date", "y"]].rename(columns={"date": "ds", "y": "y"})
    i_val = len(train) + len(val)
    WIN = 1000   # rolling history window keeps each fit fast
    prophet_pred = []
    for t in range(i_val, len(df)):
        hist = dfp.iloc[max(0, t + 1 - WIN):t + 1]
        mp = Prophet(weekly_seasonality=False, daily_seasonality=False,
                     yearly_seasonality=True, uncertainty_samples=0)
        mp.fit(hist)
        fut = mp.make_future_dataframe(periods=1, freq="B")
        prophet_pred.append(mp.predict(fut).iloc[-1]["yhat"])
    prophet_pred = np.array(prophet_pred)
    prophet_m = utils.evaluate(actual, prophet_pred)
    print("Prophet:", {k: round(v, 4) for k, v in prophet_m.items()})
    utils.save_predictions("prophet", test, prophet_pred)

11:45:49 - cmdstanpy - INFO - Chain [1] start processing


11:45:52 - cmdstanpy - INFO - Chain [1] done processing


11:45:52 - cmdstanpy - INFO - Chain [1] start processing


11:45:53 - cmdstanpy - INFO - Chain [1] done processing


11:45:53 - cmdstanpy - INFO - Chain [1] start processing


...

11:49:31 - cmdstanpy - INFO - Chain [1] start processing


11:49:32 - cmdstanpy - INFO - Chain [1] done processing


11:49:32 - cmdstanpy - INFO - Chain [1] start processing


11:49:32 - cmdstanpy - INFO - Chain [1] done processing


Prophet: {'MAE': 4.0509, 'RMSE': 5.9487, 'MAPE': 3.0677, 'R2': 0.9387}


## 5. Compare these models

In [7]:
rows = {"XGBoost": xgb_m, "LightGBM": lgb_m}
if prophet_m is not None:
    rows["Prophet"] = prophet_m
pd.DataFrame(rows).T.round(4)

,MAE,RMSE,MAPE,R2
XGBoost,1.7727,2.4528,1.3691,0.9896
LightGBM,1.4872,2.1848,1.1478,0.9917
Prophet,4.0509,5.9487,3.0677,0.9387


## Conclusion
XGBoost and LightGBM capture non-linear interactions among the features; Prophet brings a classical trend/seasonality view. The head-to-head against the random walk happens in notebook 06.